In [1]:
import torch
import torch.nn as nn

In [2]:
# ------------------------------------------------
# 1. Parameters
# ------------------------------------------------

torch.manual_seed(42)

seq_len = 5
input_size = 3
hidden_size = 4

# Input sequence
x = torch.randn(seq_len, input_size)

print("Input shape:", x.shape)

Input shape: torch.Size([5, 3])


In [3]:
# ------------------------------------------------
# 2. Create manual LSTM weights
# ------------------------------------------------

W = torch.randn(4 * hidden_size, input_size)
U = torch.randn(4 * hidden_size, hidden_size)
b = torch.randn(4 * hidden_size)

In [4]:
# ------------------------------------------------
# 3. Manual LSTM direction
# ------------------------------------------------

def lstm_direction(x, W, U, b):

    h = torch.zeros(hidden_size)
    c = torch.zeros(hidden_size)

    outputs = []

    for t in range(len(x)):

        gates = W @ x[t] + U @ h + b

        # Four LSTM gates
        i = torch.sigmoid(gates[0:hidden_size])
        f = torch.sigmoid(gates[hidden_size:2*hidden_size])
        o = torch.sigmoid(gates[2*hidden_size:3*hidden_size])
        g = torch.tanh(gates[3*hidden_size:4*hidden_size])

        # Cell state
        c = f * c + i * g

        # Hidden state
        h = o * torch.tanh(c)

        outputs.append(h)

    return torch.stack(outputs)

In [5]:

# ------------------------------------------------
# 4. Forward LSTM
# ------------------------------------------------

forward_output = lstm_direction(x, W, U, b)

print("Forward output shape:", forward_output.shape)


Forward output shape: torch.Size([5, 4])


In [6]:

# ------------------------------------------------
# 5. Backward LSTM
# ------------------------------------------------

reverse_x = torch.flip(x, [0])

backward_output = lstm_direction(
    reverse_x,
    W,
    U,
    b
)

backward_output = torch.flip(
    backward_output,
    [0]
)

print("Backward output shape:", backward_output.shape)

Backward output shape: torch.Size([5, 4])


In [7]:

# ------------------------------------------------
# 6. Bidirectional output
# ------------------------------------------------

bilstm_output = torch.cat(
    (forward_output, backward_output),
    dim=1
)

print("Bidirectional output shape:", bilstm_output.shape)

Bidirectional output shape: torch.Size([5, 8])


In [8]:
# ------------------------------------------------
# 7. Built-in PyTorch BiLSTM
# ------------------------------------------------

builtin_lstm = nn.LSTM(
    input_size=input_size,
    hidden_size=hidden_size,
    num_layers=1,
    bidirectional=True
)



In [9]:
# ------------------------------------------------
# 8. Match PyTorch gate order
# PyTorch order:
# Input, Forget, Candidate, Output
# ------------------------------------------------

order = torch.cat([
    torch.arange(hidden_size),
    torch.arange(3 * hidden_size, 4 * hidden_size),
    torch.arange(hidden_size, 2 * hidden_size),
    torch.arange(2 * hidden_size, 3 * hidden_size)
])

W_pytorch = W[order]
U_pytorch = U[order]
b_pytorch = b[order]


In [10]:

# ------------------------------------------------
# 9. Copy weights
# ------------------------------------------------

with torch.no_grad():

    # Forward
    builtin_lstm.weight_ih_l0.copy_(W_pytorch)
    builtin_lstm.weight_hh_l0.copy_(U_pytorch)
    builtin_lstm.bias_ih_l0.copy_(b_pytorch)
    builtin_lstm.bias_hh_l0.zero_()

    # Backward
    builtin_lstm.weight_ih_l0_reverse.copy_(W_pytorch)
    builtin_lstm.weight_hh_l0_reverse.copy_(U_pytorch)
    builtin_lstm.bias_ih_l0_reverse.copy_(b_pytorch)
    builtin_lstm.bias_hh_l0.zero_()

In [11]:

# ------------------------------------------------
# 10. Run built-in BiLSTM
# ------------------------------------------------

x_batch = x.unsqueeze(1)

builtin_output, (h_n, c_n) = builtin_lstm(x_batch)

builtin_output = builtin_output.squeeze(1)

print("Built-in output shape:", builtin_output.shape)


Built-in output shape: torch.Size([5, 8])


In [12]:

# ------------------------------------------------
# 11. Compare results
# ------------------------------------------------

difference = torch.abs(
    bilstm_output - builtin_output
)

print("Maximum difference:", difference.max().item())
print("Average difference:", difference.mean().item())

print(
    "Outputs match:",
    torch.allclose(
        bilstm_output,
        builtin_output,
        atol=1e-5
    )
)

Maximum difference: 1.4665133953094482
Average difference: 0.2756505012512207
Outputs match: False
